In [18]:
import pandas as pd
training_df = pd.read_csv("/kaggle/input/category-adapter/training_data.csv")
testing_df = pd.read_csv("/kaggle/input/category-adapter/testing_data.csv")
def LoadData(category):
    main_categories_keywords = {
    "robbery": (
        "robbery|theft|snatching|highway robbery|banditry|mugging|"
        "armed robbery|dacoity|burglary|looting|heist|plunder|"
        "shoplifting|pickpocketing|stealing|house robbery|carjacking|"
        "train robbery|bank robbery|street crime|robber gang|loot attempt"
    ),

    "murder": (
        "murder|homicide|killing|manslaughter|assassination|"
        "slaying|beheading|shooting to death|stabbed to death|"
        "strangulation|gunshot killing|cold-blooded murder|"
        "brutal killing|honour killing|targeted killing|lynching|"
        "poisoning death|fatal stabbing|axe attack killing"
    ),

    "sexual assault": (
        "sexual assault|rape|molestation|gang rape|harassment|"
        "sexual harassment|eve teasing|child abuse|sexual abuse|"
        "indecent assault|attempted rape|forced sex|"
        "marital rape|sexual violence|sex crime|"
        "outraging modesty|inappropriate touching|abuse of minor"
    ),

    "suicide": (
        "suicide|attempted suicide|self-harm|self immolation|"
        "took own life|ended life|committed suicide|"
        "ingested poison|drank acid|set on fire self|"
        "hung himself|hung herself|jumped from building|"
        "jumped into river|suicidal act|tied rope around neck"
    ),

    "kidnapping": (
        "kidnapping|abduction|child abduction|forced disappearance|"
        "hostage taking|held hostage|kidnap attempt|"
        "abducted minor|kidnapped child|school abduction|"
        "abducted for ransom|ransom demand|missing person|"
        "abducted woman|forced confinement|abduction plot"
    ),

    "terrorist attack": (
        "terrorist attack|terrorism|militant attack|suicide bombing|suicide blast|"
        "IED explosion|car bomb|truck bomb|roadside bomb|grenade attack|"
        "improvised explosive device|terror plot|terror bid|terror attempt|"
        "extremist violence|militant violence|armed militants|gun attack|firing incident|"
        "shootout by militants|targeted attack|terror outfit|terror network|jihadist attack|"
        "Islamist militants|insurgent attack|Taliban attack|Al-Qaeda attack|ISIS attack|"
        "sectarian attack|religious extremism|terror suspect|terror cell|"
        "terror activity|terror group|terror strike|bomb|Bomb"
    )
}
    keywords = main_categories_keywords[category]
    
    
    # print(training_df.head())
    
    category_data = training_df[training_df['arr_labels'].apply(lambda x: category in x)]
    print(category_data.shape)
    non_category_data = training_df[
        training_df['text'].str.contains(keywords, regex=True, na=False) &
        (~training_df.index.isin(category_data.index)) 
    ]
    category_df = pd.concat([ category_data, non_category_data])
    # category_df.drop_duplicates(subset=['url'], inplace=True)
    
    
    test_category_data = testing_df[testing_df['arr_labels'].apply(lambda x: category in x)]
    
    test_non_category_data = testing_df[
        testing_df['text'].str.contains(keywords, regex=True, na=False) &
        (~testing_df.index.isin(category_data.index)) 
    ]
    test_category_df = pd.concat([ test_non_category_data, test_category_data])
    print(category_df.shape)
    print(test_category_df.shape)

    return category_df, test_category_df

In [19]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

def convert_labels(raw_df1, mlb=None):
    raw_df = raw_df1.copy()

    main_categories = {
    "robbery": ["theft", "robbery", "highway robbery", "banditry"],
    "murder": ["murder"],
    "sexual assault": ["sexual assault", "rape", "molestation"],
    "suicide": ["suicide"],
    "kidnapping": ["kidnapping", "abduction"],
    "terrorist attack": ["terrorism"]
}


    def normalize_labels(labels):
        normalized = []
        for label in labels:
            label = str(label).strip() if label is not None else ""
            found = False
            if label == "0" or label == 0:
                continue
            for main_cat, variants in main_categories.items():
                if label in variants:
                    if main_cat not in normalized:
                        normalized.append(main_cat)
                    found = True
                    break
            if not found and "other_crime" not in normalized:

                normalized.append("other_crime")
        return sorted(set(normalized))

    raw_df['text_label'] = raw_df['arr_labels'].apply(normalize_labels)
    raw_df['text_label'] = raw_df['arr_labels'].apply(
    lambda labels: ["other_crime"] if "other_crime" in labels else labels
)

    all_normalized = [
        "ڈکیتی", "قتل", "جنسی زیادتی", "خود کشی",
         "اغوا", "other_crime", "دہشت گردی",
    ]

    # Fit only if mlb is not passed (for training)

    raw_df['label'] = raw_df['text_label'].apply(lambda x: 1 if "other" not in x else 0)
    print(category)

    return raw_df



In [6]:
from sklearn.model_selection import train_test_split


In [ ]:
!pip install optuna

In [ ]:
from transformers import AutoTokenizer

model_name = "google-bert/bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset

# ---------------- Metrics ----------------
def compute_metrics_single_label(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {"micro_f1": micro_f1, "macro_f1": macro_f1}

# ---------------- Training Function ----------------
def train_single_label_full_model(train_df, val_df, test_df, tokenizer, model_name,
                                  batch_size=8, max_token=512, is16=False):

    # ---- Dataset Formatting ----
    def tokenize_and_format(batch):
        tokens = tokenizer(
            batch['text'], padding="max_length", truncation=True, max_length=max_token
        )
        tokens["labels"] = batch["label"]
        return tokens

    train_dataset = Dataset.from_pandas(train_df[['text', 'label']], preserve_index=False)
    val_dataset   = Dataset.from_pandas(val_df[['text', 'label']], preserve_index=False)
    test_dataset  = Dataset.from_pandas(test_df[['text', 'label']], preserve_index=False)

    train_dataset = train_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])
    val_dataset   = val_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])
    test_dataset  = test_dataset.map(tokenize_and_format, batched=True, remove_columns=['text', 'label'])

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    # ---- Load Base Model (FULL training) ----
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=int(np.unique(train_df["label"]).shape[0]),
        problem_type="single_label_classification"
    )

    # ---- Training Arguments ----
    training_args = TrainingArguments(
        output_dir="./results_full_model",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        weight_decay=0.01,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=is16,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,           # keep only the latest/best checkpoint
        load_best_model_at_end=True,
        save_only_model=True,         # <<<<<< key fix: do NOT save optimizer/scheduler
        logging_dir="./logs_full_model",
        report_to="none",
        logging_steps=50
    )

    # ---- Trainer with Early Stopping ----
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_single_label,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # ---- Train ----
    trainer.train()

    # ---- Predictions ----
    predictions = trainer.predict(test_dataset)
    logits = predictions.predictions
    labels = np.array(predictions.label_ids)
    preds = np.argmax(logits, axis=1)

    micro_f1 = f1_score(labels, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)

    print(f"\n[INFO] Micro F1: {micro_f1:.4f}")
    print(f"[INFO] Macro F1: {macro_f1:.4f}")
    print(classification_report(labels, preds, zero_division=0))

    # ---- Save Model & Tokenizer (weights only) ----
    model.save_pretrained("./full_model_single_label")
    tokenizer.save_pretrained("./full_model_single_label")

    return model, tokenizer


In [11]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from sklearn.metrics import accuracy_score, classification_report

def predict_from_hf_model(model, test_df, category, batch_size=16):
    """
    Run predictions on a Hugging Face-hosted sequence classification model.

    Args:
        model_name (str): Hugging Face model name (e.g., "your-username/urdu-classifier").
        test_df (pd.DataFrame): DataFrame with 'text' and 'label' columns.
        category (int or str): Positive label category to evaluate (0 or 1 for binary tasks).
        batch_size (int): Number of samples per batch.

    Returns:
        tuple: (accuracy, classification_report_str)
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model.to(device)
    model.eval()

    max_len = min(tokenizer.model_max_length, 512)

    texts = test_df["text"].astype(str).tolist()
    labels = test_df["text_label"].apply(lambda x: 1 if "other" not in x else 0).astype(int).tolist()

    preds, y_true = [], []

    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            batch_labels = labels[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            logits = outputs.logits
            batch_preds = torch.argmax(logits, dim=1).cpu().numpy()

            preds.extend(batch_preds)
            y_true.extend(batch_labels)

    # Metrics
    acc = accuracy_score(y_true, preds)
    report = classification_report(
        y_true, preds,
        target_names=[f"Not {category}", str(category)],
        zero_division=0
    )

    print(f"\n[INFO] Model: {model_name}")
    print(f"[INFO] Accuracy: {acc:.4f}")
    print(f"[INFO] Classification Report:\n{report}")

    return acc, report



In [ ]:
categories = ["robbery", "suicide", "kidnapping", "terrorist attack", 
              "sexual assault", "murder", "other"]
categories = [  "event"]

for category in categories:
    # train_df, test_df = LoadData(category)
    
    category_df = convert_labels(training_df)
    test_df = convert_labels(testing_df)
    train_df, val_df = train_test_split(
        category_df,
        test_size=0.1,
        random_state=42,
        shuffle=True
    )
    def drop_invalid_rows(df, col="text"):
        # Keep only rows where column is a non-empty string
        mask = df[col].apply(lambda x: isinstance(x, str) and x.strip() != "")
        return df[mask].copy()
    
    # Apply to all
    train_df = drop_invalid_rows(train_df, "text")
    val_df   = drop_invalid_rows(val_df, "text")
    test_df  = drop_invalid_rows(test_df, "text")
    model, tokenizer = train_single_label_full_model(train_df, test_df,  val_df, tokenizer, model_name,
            is16=False, batch_size=8)
    results = predict_from_hf_model(
        model=model, 
        test_df=test_df,
        category =  category
    )
    model.push_to_hub(f"the-usan/eng-crime-{category.replace(' ', '_')}-v3")
    tokenizer.push_to_hub(f"the-usan/eng-crime-{category.replace(' ', '_')}-v3")


In [ ]:
model, tokenizer = train_single_label_full_model(train_df, test_df,  val_df, tokenizer, model_name,
            is16=False, batch_size=8)

In [ ]:


print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
